# AgentOps Lab 03 - Rebuild the agent using OpenAI Agents SDK

In Notebook 1, you built the loop yourself. In Notebook 2, you learned when not to use an agent. Now you rebuild the incident investigator using a framework-shaped design.

The OpenAI Agents SDK is useful when you want a runtime to manage turns, tool execution, guardrails, handoffs, sessions, and tracing. The lesson is subtle but important: frameworks do not remove the agent loop. They package it.



## Notebook-first learning contract

This notebook is the primary lesson for this topic. The Python module is not a separate replacement for the lesson; it is the implementation layer that the notebook explains, runs, breaks, and evaluates. Work through the notebook in this order:

1. read the concept model and architecture boundary;
2. inspect the tool/state/policy contracts;
3. run the deterministic implementation;
4. trigger the deliberate failure case;
5. record evaluation, cost, latency, and safety observations; and
6. answer the architecture question before moving on.


## Deep-dive training guide — Managed single agent with OpenAI Agents SDK

### Concepts to master

- framework-managed loop versus application-owned policy
- tools, sessions, tracing, and guardrails
- what an SDK packages and what it cannot decide for you

### Implementation walkthrough

Compare the manual trace with the SDK-shaped trace in `agents_sdk_rebuild.py`. The lab models Agent, Runner, function tools, trace events, and session state without requiring credentials.

### Deliberate failure case

Give the SDK-shaped agent a vague tool or remove the max-turn guard. The framework still runs the loop; it does not magically fix unsafe tool design or missing stop rules.

### Learner exercise

Port one read-only tool to a real `@function_tool` if you have credentials, then compare the trace shape with the deterministic teaching double.

### What to write down

For each run, capture the chosen architecture, tool trajectory, evidence used, rejected alternatives, stop condition, estimated cost, latency, and one sentence explaining whether the architecture was the least autonomous reliable option.


## Engineering checklist for this notebook

Use this checklist as your mini design review before you call the topic complete.

| Area | Question to answer |
| --- | --- |
| Control boundary | Which decisions are made by deterministic code, and which are delegated to the model? |
| Tools | Are tool inputs typed, narrow, authorized, and auditable? |
| State | What state is carried between steps, and what should never become long-term memory? |
| Failure mode | What is the easiest way this design loops, overacts, or fabricates certainty? |
| Evaluation | Which outcome, trajectory, safety, cost, and latency signals prove the design is working? |
| Architecture choice | Why is this architecture simpler or better than the nearest alternative? |


## What moves into the framework?

```mermaid
flowchart LR
    A["Manual loop"] --> B["Application owns messages"]
    A --> C["Application dispatches tools"]
    A --> D["Application records trace"]
    E["Agents SDK"] --> F["Runner manages turns"]
    E --> G["Function tools expose schemas"]
    E --> H["Tracing records model and tool spans"]
    E --> I["Sessions preserve working context"]
```

You still own the product boundary: which tools exist, what they are allowed to do, which actions require approval, what evidence is sufficient, and what counts as safe completion.


## Real SDK shape

The real implementation looks like this. This cell is shown as reference because it requires `openai-agents` and `OPENAI_API_KEY`.

```python
from agents import Agent, Runner, function_tool

@function_tool
def get_service_status(service: str) -> dict:
    ...

@function_tool
def search_incidents(query: str) -> list:
    ...

@function_tool
def get_runbook(service: str) -> str:
    ...

incident_agent = Agent(
    name="Incident Investigator",
    instructions="""
    Investigate operational incidents.
    Always gather evidence before diagnosing a problem.
    Use the minimum number of tools required.
    Never execute remediation actions.
    """,
    tools=[get_service_status, search_incidents, get_runbook],
)

result = await Runner.run(incident_agent, "European users report checkout failures.")
print(result.final_output)
```

The SDK docs describe Agents as models equipped with instructions and tools; function tools turn Python functions into tools with schema generation; sessions carry working context; tracing records model calls, tool calls, guardrails, handoffs, and custom events.


In [ ]:
from pathlib import Path
import sys

repo_root = next((candidate for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (candidate / "curriculum" / "shared" / "agentops_lab").exists()), None)
if repo_root is None:
    raise RuntimeError("Run this notebook from inside the repository checkout.")
sys.path.insert(0, str(repo_root / "curriculum" / "shared"))

from agentops_lab.agents_sdk_rebuild import OfflineAgentsSDKRuntime, TOOLS, compare_manual_and_framework


## SDK ownership matrix

A framework changes who writes orchestration code, not who owns product safety. Use this matrix to avoid both extremes: rebuilding everything by hand forever, or assuming the SDK makes architecture decisions for you.

| Concern | Manual loop | OpenAI Agents SDK-shaped runtime | Application still owns |
| --- | --- | --- | --- |
| Tool schema | Hand-written dispatch contract | Function tools expose structured schemas | Tool scope, risk tier, authorization |
| Turn loop | You write the loop | Runner manages turns | Budgets, escalation policy, success criteria |
| Tool execution | You call functions | Runtime dispatches tool calls | Which tools exist and whether side effects are allowed |
| Trace | You format logs | Tracing records model/tool/guardrail spans | What to evaluate and alert on |
| Sessions | You persist messages | Session abstraction can preserve context | Memory policy, retention, privacy |
| Guardrails | You wrap checks manually | Agent/tool guardrail hooks can package checks | The actual policy and failure behavior |

The productive mindset is: let the framework remove boilerplate, then spend the saved attention on better tools, policies, evals, and traces.


In [ ]:
comparison = compare_manual_and_framework()
print("Manual final answer:
", comparison["manual"]["final_output"])
print("
Framework-shaped final answer:
", comparison["framework"]["final_output"])


## Trace analysis: what would you inspect in production?

The SDK-shaped trace includes session, model, tool schema, tool call, guardrail, and final-answer events. In a real incident assistant, these spans are the evidence you use to debug regressions and review release readiness.


In [ ]:
from collections import Counter

events = comparison["framework"]["trace"]
print(Counter(event["kind"] for event in events))
for event in events:
    print(f"{event['kind']:12} {event['name']:24} {str(event['detail'])[:140]}")


## Guardrail thought experiment

The framework can run guardrails, but you still have to define the policy. For this incident investigator, a useful first guardrail is **evidence required before diagnosis**. Another is **no remediation execution**: the agent may prepare a rollback recommendation, but not call rollback or restart tools.

Add these to your design review:

- What minimum evidence is required before claiming an active incident?
- Which tool calls are read-only?
- Which tool calls are propose-only?
- Which calls require human approval?
- What trace span proves the guardrail ran?


In [ ]:
guardrail_events = [event for event in events if event["kind"] == "guardrail"]
assert guardrail_events, "Expected at least one guardrail event"
assert all(event["detail"].get("passed") for event in guardrail_events), guardrail_events
print("Guardrails passed:", guardrail_events)


## Porting exercise: from teaching double to real SDK

When you port this notebook to the real SDK, do not start by adding powerful tools. Start with the three read-only tools from Notebook 1. Keep the same evaluation question: did the framework produce a supported recommendation with fewer lines of orchestration code while preserving policy boundaries?

Suggested porting steps:

1. Wrap `get_service_status`, `search_incidents`, and `get_runbook` as function tools.
2. Create an `Incident Investigator` agent with evidence-first instructions.
3. Run one incident prompt through `Runner.run`.
4. Inspect trace events for model calls, tool calls, and guardrails.
5. Compare the trajectory with the manual loop.
6. Add a test that fails if a write/remediation tool appears in the trajectory.


## Run the offline framework-shaped version

This teaching double keeps the notebook runnable without credentials. It mirrors the responsibilities you would inspect in a framework run: session start, model planning, tool schema/dispatch, guardrail outcome, and final response.


In [ ]:
comparison = compare_manual_and_framework()
comparison["framework"]["final_output"]


In [ ]:
print("Manual owns:")
for item in comparison["manual"]["owned_by_application"]:
    print("-", item)

print("\nFramework owns:")
for item in comparison["framework"]["framework_owns"]:
    print("-", item)


## Inspect the trace

A trace should answer: what did the agent decide, which tools were exposed, which calls were made, what observations came back, which guardrails passed, and why did the run stop?


In [ ]:
for event in comparison["framework"]["trace"]:
    print(f"{event['kind']:12} {event['name']}")


## Optional real SDK experiment

Install `openai-agents`, set `OPENAI_API_KEY`, and port the deterministic tools into `@function_tool` functions. Keep the tools read-only for this notebook. Then compare the real trace against the offline trace above.

Questions to answer:

- Which parts of the manual loop disappeared from your code?
- Which safety decisions still live in your application?
- Did the framework reduce code, or did it move the code into configuration?
- What trace span would you inspect first if the agent overused tools?


## Takeaway

The SDK packages the loop, schemas, dispatch, messages, sessions, and tracing. It does not choose your product boundary for you. You still design tools, permissions, evidence rules, budgets, approval gates, and evaluation.

References: [OpenAI Agents SDK](https://openai.github.io/openai-agents-python/), [Agents SDK tools](https://openai.github.io/openai-agents-python/tools/), [Agents SDK tracing](https://openai.github.io/openai-agents-python/tracing/), and [Agents SDK sessions](https://openai.github.io/openai-agents-python/sessions/).
